# 04 · Robustez y significancia

## Paso 7 · ¿Habilidad o suerte?

| Prueba | Pregunta | Cómo leerla |
|---|---|---|
| Bootstrap por bloques del Sharpe | ¿Qué tan incierto es mi Sharpe? | Si el IC 95 % incluye 0, no hay evidencia de ventaja |
| Bootstrap pareado vs. buy & hold | ¿La diferencia contra el benchmark es real? | Si el IC de la diferencia incluye 0, no se distingue |
| Deflated Sharpe Ratio | ¿Sigue siendo bueno tras probar 32 combinaciones? | > 0.95 evidencia razonable; 0.5-0.95 ambiguo; < 0.5 compatible con azar |
| Placebo de timing | ¿La señal aporta, o solo gano por estar invertido? | Si el timing aleatorio rinde igual, la señal no sirve |
| Mapa de parámetros | ¿Meseta o pico aislado? | Meseta = robusta; pico = sobreajuste |

Se parte de las series fuera de muestra del Paso 6 (walk-forward 2010-2025), que se recalculan aquí (tardan ~1 s).

In [1]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.data import download_prices
from src.signals import STRATEGIES, sma_cross
from src.backtest import run_backtest
from src.metrics import sharpe
from src.validation import (GRIDS, param_combos, walk_forward, block_bootstrap_sharpe,
                            paired_bootstrap_diff, deflated_sharpe, expected_max_sharpe,
                            psr, timing_placebo)
sns.set_theme(style="whitegrid")

px = download_prices(cache="../data/raw/prices.parquet")
start = px.index[260]
oos, chosen = {}, {}
for name in GRIDS:
    oos[name], chosen[name] = walk_forward(px, STRATEGIES[name], GRIDS[name])
oos_start = oos["sma_cross"].index[0]
bench = run_backtest(px, STRATEGIES["buy_and_hold"](px), start=oos_start)
print("Fuera de muestra:", oos_start.date(), "a", bench.index[-1].date(), f"({len(bench)} días)")

Fuera de muestra: 2010-01-04 a 2025-12-30 (4023 días)


### 7.2 Intervalos de confianza fuera de muestra (bootstrap por bloques de 20 días, 2,000 remuestras)

In [2]:
rows = []
lo, hi = block_bootstrap_sharpe(bench["net"])
rows.append({"estrategia": "buy_and_hold", "sharpe": sharpe(bench["net"]), "ic95_bajo": lo, "ic95_alto": hi})
for n in GRIDS:
    lo, hi = block_bootstrap_sharpe(oos[n]["net"])
    (dlo, dhi), p = paired_bootstrap_diff(oos[n]["net"], bench["net"])
    rows.append({"estrategia": n, "sharpe": sharpe(oos[n]["net"]), "ic95_bajo": lo, "ic95_alto": hi,
                 "dif_vs_bh": sharpe(oos[n]["net"]) - sharpe(bench["net"]),
                 "dif_ic95_bajo": dlo, "dif_ic95_alto": dhi, "P(no mejor que bh)": p})
ci = pd.DataFrame(rows).set_index("estrategia")
ci.round(2)

,sharpe,ic95_bajo,ic95_alto,dif_vs_bh,dif_ic95_bajo,dif_ic95_alto,P(no mejor que bh)
estrategia,,,,,,,
buy_and_hold,0.92,0.44,1.41,NaN,NaN,NaN,NaN
sma_cross,0.82,0.37,1.33,-0.10,-0.38,0.20,0.73
momentum,0.73,0.28,1.21,-0.19,-0.50,0.11,0.89
mean_reversion,0.32,-0.06,0.79,-0.60,-0.95,-0.22,1.00
vol_target,0.95,0.47,1.46,0.02,-0.20,0.23,0.42


### 7.3 Deflated Sharpe Ratio (32 intentos, según `EXPERIMENTS.md`)

In [3]:
trial_sr = []
for name, grid in GRIDS.items():
    for p in param_combos(grid):
        res = run_backtest(px, STRATEGIES[name](px, **p), start=oos_start)
        trial_sr.append(res["net"].mean() / res["net"].std())      # Sharpe por día
n_trials, var_sr = len(trial_sr), np.var(trial_sr, ddof=1)
sr0 = expected_max_sharpe(n_trials, var_sr)
print(f"intentos: {n_trials} | Sharpe máximo esperado solo por azar: {sr0 * np.sqrt(252):.2f} anual")

dsr = pd.DataFrame({n: {"sharpe_anual": sharpe(oos[n]["net"]),
                        "PSR (vs 0)": psr(oos[n]["net"]),
                        "DSR (32 intentos)": deflated_sharpe(oos[n]["net"], n_trials, var_sr)}
                    for n in GRIDS}).T
dsr.round(3)

intentos: 32 | Sharpe máximo esperado solo por azar: 0.59 anual


,sharpe_anual,PSR (vs 0),DSR (32 intentos)
sma_cross,0.822,0.999,0.817
momentum,0.732,0.998,0.712
mean_reversion,0.319,0.898,0.141
vol_target,0.948,1.000,0.921


### 7.4 Placebo de timing (estrategias que entran y salen del mercado; parámetros fijos, 2006-2025, 500 simulaciones)
No aplica a buy & hold ni a la volatilidad objetivo, que siempre están invertidas.

In [4]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
placebo = {}
for ax, name in zip(axes, ["sma_cross", "momentum", "mean_reversion"]):
    real, sims, pval = timing_placebo(px, STRATEGIES[name](px), start=start)
    placebo[name] = {"sharpe_real": real, "placebo_medio": sims.mean(), "placebo_p95": np.percentile(sims, 95), "p_valor": pval}
    ax.hist(sims, bins=40, alpha=0.7)
    ax.axvline(real, color="red", label=f"real = {real:.2f}")
    ax.set_title(f"{name}: p = {pval:.3f}"); ax.legend()
fig.suptitle("Señal real vs. timing aleatorio (misma exposición y rotación)")
plt.tight_layout()
plt.savefig("../reports/figures/09_timing_placebo.png", dpi=150, bbox_inches="tight")
pd.DataFrame(placebo).T.round(3)

,sharpe_real,placebo_medio,placebo_p95,p_valor
sma_cross,0.793,0.661,0.954,0.216
momentum,0.788,0.648,0.941,0.216
mean_reversion,0.278,0.142,0.449,0.224


**Robustez del placebo a la semilla aleatoria.** El cruce de medias y el momentum dan el mismo p-valor con la semilla 42 por coincidencia (108 de 500 placebos ≥ real en ambos): sus simulaciones son distintas pero muy correlacionadas, porque las dos son señales de tendencia y usan los mismos desplazamientos. Con otras semillas:

In [5]:
pd.DataFrame({f"semilla {s}": {n: timing_placebo(px, STRATEGIES[n](px), start=start, seed=s)[2]
                                for n in ["sma_cross", "momentum", "mean_reversion"]}
              for s in [42, 1, 7]}).round(3)

,semilla 42,semilla 1,semilla 7
sma_cross,0.216,0.180,0.214
momentum,0.216,0.190,0.192
mean_reversion,0.224,0.216,0.246


### 7.5 Mapa de parámetros (cruce de medias, 2006-2025)

In [6]:
rows = []
for p in param_combos(GRIDS["sma_cross"]):
    res = run_backtest(px, sma_cross(px, **p), start=start)
    rows.append({**p, "sharpe": sharpe(res["net"])})
land = pd.DataFrame(rows).pivot(index="fast", columns="slow", values="sharpe")

plt.figure(figsize=(6, 3.5))
sns.heatmap(land, annot=True, fmt=".2f", cmap="RdYlGn")
plt.title("Sharpe según parámetros (cruce de medias)")
plt.tight_layout()
plt.savefig("../reports/figures/10_param_landscape.png", dpi=150)
print(f"rango de Sharpe en la rejilla: {land.min().min():.2f} a {land.max().max():.2f}")
land.round(2)

rango de Sharpe en la rejilla: 0.74 a 0.89


slow,100,150,200,250
fast,,,,
20,0.84,0.82,0.84,0.82
50,0.84,0.86,0.79,0.83
100,NaN,0.74,0.79,0.89


### 7.6 Veredicto

**Ninguna de las cuatro estrategias muestra evidencia de ventaja sobre buy & hold después de costos, fuera de muestra y corrigiendo por los 32 intentos.**

- **Contra buy & hold, ninguna se distingue.** El IC 95 % de la diferencia de Sharpe incluye 0 en cruce de medias [−0.38, +0.20], momentum [−0.50, +0.11] y volatilidad objetivo [−0.20, +0.23]. La reversión a la media es **significativamente peor** [−0.95, −0.22].
- **Sharpe "bueno" ≠ habilidad.** Volatilidad objetivo (DSR 0.92) y cruce de medias (DSR 0.82) tienen Sharpe positivo más allá de lo que daría el mejor de 32 intentos al azar (≈ 0.59), pero su DSR queda bajo 0.95: evidencia ambigua. Momentum (0.71) es ambiguo y reversión a la media (0.14) es compatible con puro azar. Además, buy & hold solo ya tiene Sharpe 0.92 en este periodo alcista: la mayor parte del Sharpe viene de **estar invertido**, no de la señal.
- **Placebo de timing:** con la misma exposición y rotación pero en momentos aleatorios, el ~20 % de los placebos iguala o supera a la señal real (p ≈ 0.18-0.25 en las tres estrategias y con tres semillas). La señal aporta algo sobre el promedio del placebo (+0.13 de Sharpe), pero **no es estadísticamente distinguible del azar**.
- **Robustez de parámetros:** el cruce de medias es una meseta (Sharpe entre 0.74 y 0.89 en toda la rejilla, rango menor al error estándar ≈ 0.22): no depende de un parámetro afortunado, pero tampoco hay un "mejor" parámetro real.
- **Lo único robusto es la reducción de riesgo:** volatilidad objetivo y las reglas de tendencia recortan la caída máxima a la mitad aproximadamente. Es un beneficio de gestión de riesgo, no una fuente de retorno extra.
